# Modelling - SEMMA

In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

## SAMPLE (Amostragem)

### Extraindo  os Dados do Banco

In [ ]:
DB_PRINCIPAL = "../../data/processed/feature_store.db"   
DB_DIVIDAS   = "../../data/processed/database.db"         

query = Path("target.sql").read_text(encoding="utf-8")

con = sqlite3.connect(DB_PRINCIPAL)

try:
    con.execute(f"ATTACH DATABASE '{DB_DIVIDAS}' AS dividas")
    df = pd.read_sql_query(query, con)
finally:
    con.close()

print(df.shape)
print(df["target_ever90_mob12"].mean())

(104623, 299)
0.0347820268965715


In [3]:
df.to_parquet('../../data/processed/abt.parquet')

### Split de Treinamento

In [3]:
df = pd.read_parquet('../../data/processed/abt.parquet')

In [6]:
df['data_ref'] = pd.to_datetime(df['data_ref'])

data_corte = '2023-06-01'

df_train = df[df['data_ref'] < data_corte].copy()
df_oot = df[df['data_ref'] >= data_corte].copy()

print(f"Treino: {df_train.shape}")
print(f"OOT: {df_oot.shape}")

print(f"Treino: {df_train['data_ref'].min()} até {df_train['data_ref'].max()}")
print(f"OOT: {df_oot['data_ref'].min()} até {df_oot['data_ref'].max()}")

Treino: (82836, 299)
OOT: (21787, 299)
Treino: 2017-02-04 00:00:00 até 2023-05-31 00:00:00
OOT: 2023-06-01 00:00:00 até 2024-03-05 00:00:00


In [7]:
X_train = df_train.drop(columns=['target_ever90_mob12'])
y_train = df_train['target_ever90_mob12']

X_oot = df_oot.drop(columns=['target_ever90_mob12'])
y_oot = df_oot['target_ever90_mob12']